In [35]:
import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

PATH_DATA = "../data/"

In [36]:
df_proshlyi = pd.read_excel(
    PATH_DATA + "Нагрузка.xlsx",
    sheet_name='общее',
)

In [37]:
exclude_patterns = [
    'диплом', 'руководство', 'магистр', 'аспирант',
    'преддипломная', 'практика', 'ига', 'гос. экзамен',
    'идпо', 'комисс', 'секрет', 'председ', 'нормоконтрол',
    'резерв', 'отчисл'
]

def is_numeric_or_empty(value):
    if pd.isna(value):
        return True
    try:
        pd.to_numeric(value)
        return True
    except (ValueError, TypeError):
        return False

# СОЗДАЕМ МАСКУ ДЛЯ ФИЛЬТРАЦИИ
mask = (
    df_proshlyi['ФИО'].notna() &
    ~df_proshlyi['ФИО'].str.lower().str.contains('э-|ю-|комисс|резерв|пустое|отчисл', na=False, regex=True) &
    ~df_proshlyi['Вид нагрузки'].str.lower().str.contains('|'.join(exclude_patterns), na=False, regex=True) &
    df_proshlyi['Нагрузка'].apply(is_numeric_or_empty)
)

# ПОЛУЧАЕМ ТОЛЬКО ОБЫЧНУЮ НАГРУЗКУ
df_regular = df_proshlyi[mask].copy()

print(f"Всего строк в исходном файле: {len(df_proshlyi)}")
print(f"Строк с обычной нагрузкой: {len(df_regular)}")

Всего строк в исходном файле: 1164
Строк с обычной нагрузкой: 909


In [38]:
fio_dict = {}
for _, row in df_regular.iterrows():
    key = (str(row['Дисциплина']).strip(), str(row['Вид нагрузки']).strip(), str(row['Группы']).strip())
    fio = str(row['ФИО']).strip()
    if key not in fio_dict:
        fio_dict[key] = []
    if fio and fio not in fio_dict[key]:
        fio_dict[key].append(fio)

print(f"Найдено {len(fio_dict)} уникальных сочетаний (Дисциплина, Вид нагрузки, Группы)")

Найдено 860 уникальных сочетаний (Дисциплина, Вид нагрузки, Группы)


In [39]:
print("\nОткрываем исходный файл...")
wb = openpyxl.load_workbook(PATH_DATA + "Нагрузка.xlsx")
ws = wb['общее']

wb_new = openpyxl.Workbook()
ws_new = wb_new.active
ws_new.title = 'общее'


Открываем исходный файл...


c:\Users\111\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


In [40]:
print("Копируем заголовки...")
for col in range(1, ws.max_column + 1):
    src_cell = ws.cell(1, col)
    dst_cell = ws_new.cell(1, col)
    dst_cell.value = src_cell.value
    if src_cell.has_style:
        dst_cell.font = src_cell.font.copy()
        dst_cell.fill = src_cell.fill.copy()
        dst_cell.border = src_cell.border.copy()
        dst_cell.alignment = src_cell.alignment.copy()
        dst_cell.number_format = src_cell.number_format

Копируем заголовки...


C:\Users\111\AppData\Local\Temp\ipykernel_14960\3366881158.py:7: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.font = src_cell.font.copy()
C:\Users\111\AppData\Local\Temp\ipykernel_14960\3366881158.py:8: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.fill = src_cell.fill.copy()
C:\Users\111\AppData\Local\Temp\ipykernel_14960\3366881158.py:9: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.border = src_cell.border.copy()
C:\Users\111\AppData\Local\Temp\ipykernel_14960\3366881158.py:10: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.alignment = src_cell.alignment.copy()


In [41]:
headers = [ws.cell(1, col).value for col in range(1, ws.max_column + 1)]
col_idx = {h: i+1 for i, h in enumerate(headers)}

col_disc = col_idx.get('Дисциплина', 2)
col_vid = col_idx.get('Вид нагрузки', 3)
col_groups = col_idx.get('Группы', 4)
col_fio = col_idx.get('ФИО', 9)

print(f"Столбцы: Дисциплина={col_disc}, Вид нагрузки={col_vid}, Группы={col_groups}, ФИО={col_fio}")

Столбцы: Дисциплина=2, Вид нагрузки=3, Группы=4, ФИО=9


In [42]:
row_count = 0
filled_count = 0

col_semestr = col_idx.get('семестр', 1)

# Определяем максимальную строку с данными (до появления служебных формул)
max_data_row = 0
for src_row in range(2, ws.max_row + 1):
    semestr = str(ws.cell(src_row, col_semestr).value or "").strip()
    if semestr != "":
        max_data_row = src_row
    else:
        # Как только встретили пустой семестр - дальше служебные данные
        break

print(f"Последняя строка с данными: {max_data_row}")

for src_row in range(2, max_data_row + 1):  # <-- КОПИРУЕМ ТОЛЬКО ДО max_data_row
    # Проверяем, есть ли данные в строке
    if ws.cell(src_row, 1).value is None and ws.cell(src_row, col_disc).value is None:
        continue
    
    # Получаем значения для проверки
    disc = str(ws.cell(src_row, col_disc).value or "").strip()
    vid = str(ws.cell(src_row, col_vid).value or "").strip()
    groups = str(ws.cell(src_row, col_groups).value or "").strip()
    fio_current = str(ws.cell(src_row, col_fio).value or "").strip()
    load = ws.cell(src_row, col_idx.get('Нагрузка', 5)).value
    
    # Проверяем, подходит ли строка под маску
    has_fio = fio_current != "" and fio_current != "None"
    
    is_service_fio = False
    service_patterns = ['э-', 'ю-', 'комисс', 'резерв', 'пустое', 'отчисл']
    for pattern in service_patterns:
        if pattern in fio_current.lower():
            is_service_fio = True
            break
    
    is_excluded_vid = False
    for pattern in exclude_patterns:
        if pattern in vid.lower():
            is_excluded_vid = True
            break
    
    is_numeric_load = True
    if load is not None:
        try:
            pd.to_numeric(load)
        except (ValueError, TypeError):
            is_numeric_load = False
    
    # Если строка не проходит фильтр - пропускаем
    if not has_fio or is_service_fio or is_excluded_vid or not is_numeric_load:
        continue
    
    # Копируем только если прошла фильтр
    dst_row = row_count + 2
    
    for col in range(1, ws.max_column + 1):
        src_cell = ws.cell(src_row, col)
        dst_cell = ws_new.cell(dst_row, col)
        dst_cell.value = src_cell.value
        if src_cell.has_style:
            dst_cell.font = src_cell.font.copy()
            dst_cell.fill = src_cell.fill.copy()
            dst_cell.border = src_cell.border.copy()
            dst_cell.alignment = src_cell.alignment.copy()
            dst_cell.number_format = src_cell.number_format
    
    # Заполняем ФИО из словаря (если есть)
    key = (disc, vid, groups)
    if key in fio_dict and fio_dict[key]:
        ws_new.cell(dst_row, col_fio).value = fio_dict[key][0]
        filled_count += 1
    
    row_count += 1

print(f"Скопировано строк с обычной нагрузкой: {row_count}")
print(f"Заполнено ФИО: {filled_count}")

Последняя строка с данными: 1130


C:\Users\111\AppData\Local\Temp\ipykernel_14960\3099284059.py:65: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.font = src_cell.font.copy()
C:\Users\111\AppData\Local\Temp\ipykernel_14960\3099284059.py:66: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.fill = src_cell.fill.copy()
C:\Users\111\AppData\Local\Temp\ipykernel_14960\3099284059.py:67: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.border = src_cell.border.copy()
C:\Users\111\AppData\Local\Temp\ipykernel_14960\3099284059.py:68: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  dst_cell.alignment = src_cell.alignment.copy()


Скопировано строк с обычной нагрузкой: 909
Заполнено ФИО: 909


In [43]:
print("\nНастраиваем ширину столбцов...")
for col in range(1, ws_new.max_column + 1):
    max_length = 0
    col_letter = get_column_letter(col)
    for row in range(1, min(ws_new.max_row + 1, 50)):
        cell = ws_new.cell(row, col)
        if cell.value:
            try:
                length = len(str(cell.value))
                if length > max_length:
                    max_length = length
            except:
                pass
    adjusted_width = min(max(max_length + 2, 10), 50)
    ws_new.column_dimensions[col_letter].width = adjusted_width


Настраиваем ширину столбцов...


In [44]:
output_file = 'новый_сводный_файл_2026_с_оформлением.xlsx'
wb_new.save(PATH_DATA + output_file)
print(f"\nРезультат сохранен в: {output_file}")
print(f"Всего строк в новом файле: {row_count}")


Результат сохранен в: новый_сводный_файл_2026_с_оформлением.xlsx
Всего строк в новом файле: 909
